# Workspace
エージェント機能を持つチャットボットのサンプル。 mlflow と連携

In [ ]:
import mlflow
import zoneinfo

tz_info = zoneinfo.ZoneInfo("Asia/Tokyo")
mlflow.set_experiment("agent-rag")

# Playground

In [ ]:
import datetime
from mlflow.types.agent import ChatAgentMessage
from agent_assistant import agent

ymd = datetime.datetime.now(tz=tz_info).strftime("%Y%m%d_%H%M%S")
prompt = """
去年の6～12月に何をやっていたか振り返りたい。対象時期のノートを収集してまとめて。
まとめる際は継続的に取り組んでいた事と、時期ごとの1～数か月の短期集中で取り組んでいた事に分けて扱う。
"""
mlflow.autolog()
with mlflow.start_run(run_name=f"dev_{ymd}"):
    # 推論 (逐次)
    res = agent.agent_wrapped.predict(
        messages=[ChatAgentMessage(role="user", content=prompt)]
    )
    print(res.messages)

    # # 推論 (ストリーミング)
    # for res in agent.agent_wrapped.predict_stream(
    #     messages=[ChatAgentMessage(role="user", content=prompt)]
    # ):
    #     print(res.delta)

# Test
WIP: ハイパーパラメータを設定する仕組みが必要。

- ハイパーパラメータの agent への入力
- mlflow.log_params による記録

In [ ]:
import datetime
import importlib
from agent_assistant import agent, evaluate

# テストデータを作成
eval_dataset = [
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "東京都の明日の天気を教えてください。"}
            ]
        },
        "expectations": {"expected_response": "東京都の明日の天気は晴れです"},
    },
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "横浜市の明日の天気を教えてください。"}
            ]
        },
        "expectations": {"expected_response": "横浜市の明日の天気は晴れです"},
    },
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "群馬県の明日の天気を教えてください。"}
            ]
        },
        "expectations": {"expected_response": "群馬県の天気は豪雨です。"},
    },
]

ymd = datetime.datetime.now(tz=tz_info).strftime("%Y%m%d_%H%M%S")
with mlflow.start_run(run_name=f"dev_{ymd}"):
    # モデルを評価
    importlib.reload(agent)
    res = evaluate.eval_responses(agent.agent_wrapped, eval_dataset)
    print(res)

# Launch AgentServer

`uv run serve` を起動した状態で実行。

In [ ]:
import json
import urllib.request as req

res = req.urlopen(
    "http://localhost:8000/invocations",
    json.dumps({"messages": [{"role": "user", "content": "こんにちは"}]}).encode(),
)
res = json.load(res)
res